# 05. Multi-step generation (stage 6.1)

Task question 4 and plan §8 (`docs/codeagents/research_plan.md`): what happens when the recommender generates five tracks in a row, each appended to the context as if the user had heard it. Run `04_rl` first: the policies are read from `runs/` on Drive.

`recsys_rl.generate.rollout_eval` runs every policy in three feedback modes for the appended listen (the model reads feedback tokens and has no "unknown" one):

- `accept`: played to the end, no like — every recommendation taken as consumed (plan mode (a));
- `expected`: the proxy RM's expected played ratio (mode (b));
- `sampled`: skip / partial / Listen+ and like / dislike drawn from the proxy RM (mode (c), a stochastic user simulator);

and two decodings: `greedy` (repeats inside the slate are not masked, so loops show up) and `sample`. Per step: relevance of the step's top-100 against the real Listen+ of the test day, KL to π_ref on the same extended context, entropy, what the generated track is (heard before, already in the slate, length, popularity, distinct artists in the slate) and the proxy / gold RM view of it with the ensemble spread (does the RM extrapolate more with every step?). Step 1 is the ordinary test evaluation.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import torch

from recsys_rl.config import load_config
from recsys_rl.data import prepare
from recsys_rl.generate import rollout_eval
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml")
prepared = prepare(cfg)
RESULTS = cfg.results_dir / "stage6"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)
runs = cfg.run_dir.parent

## Rollouts

Policies span the KL axis of stage 5: π_ref, RLOO R1 β = 0.1 (KL 0.14, the best RL point), RLOO R1 β = 0.02 (KL 1.3), GRPO β = 0.1 (KL 6, strong hacking: short tracks, low entropy) the distilled policy `distill` β = 0.05 (the best single model, observations §9) and the reranker R1 β = 0.1 recomputed on the extended context at every step (the RM sees the generated history too). Per-user tables are cached on Drive per policy (`runs/multistep/`), so a restarted session or a new policy only computes what is missing.

In [ ]:
POLICIES = {
    "pi_ref": None,
    "rl_ratio_b0.1": runs / "rl_ratio_b0.1" / "policy.pt",
    "rl_ratio_b0.02": runs / "rl_ratio_b0.02" / "policy.pt",
    "rl_ratio_grpo_b0.1": runs / "rl_ratio_grpo_b0.1" / "policy.pt",
    "rl_distill_b0.05": runs / "rl_distill_b0.05" / "policy.pt",  # the best single policy
}
RERANK = {"rr_ratio_b0.1": 0.1}
STEPS = 5
cache_dir = runs / "multistep"
cache_dir.mkdir(parents=True, exist_ok=True)
specs = {
    **{name: {"policies": {name: path}} for name, path in POLICIES.items()},
    **{name: {"rerank": {name: beta}} for name, beta in RERANK.items()},
}
parts = []
for name, spec in specs.items():  # one cache file per policy: new policies add to finished ones
    cache = cache_dir / f"rollouts_test_{name}.parquet"
    if not cache.exists():
        rollout_eval(cfg, prepared, device, split="test", steps=STEPS, **spec).write_parquet(cache)
    parts.append(pl.read_parquet(cache))
rollouts = pl.concat(parts).with_columns(
    gold_minus_proxy=pl.col("gold_ratio") - pl.col("proxy_ratio")
)
summary = (
    rollouts.group_by("policy", "mode", "decoding", "step", maintain_order=True)
    .agg(pl.exclude("user").cast(pl.Float64).mean())
    .sort("policy", "mode", "decoding", "step")
)
summary.write_csv(RESULTS / "rollouts_summary.csv")
summary.filter(pl.col("step").is_in([1, STEPS])).select(
    "policy",
    "mode",
    "decoding",
    "step",
    "ndcg@10",
    "hit",
    "kl_ref",
    "entropy",
    "in_history",
    "in_slate",
    "length",
    "slate_artists",
    "proxy_ratio",
    "gold_ratio",
    "gold_ratio_std",
)

## Metrics by step

One figure per decoding; colour = policy, line style = feedback mode. Step 1 is the same in all modes (nothing appended yet).

In [ ]:
PANELS = [
    ("ndcg@10", "NDCG@10 of the step's top-100 (real test Listen+)"),
    ("recall@100", "Recall@100"),
    ("hit", "generated track in the test Listen+"),
    ("kl_ref", "KL(π || π_ref) on the extended context"),
    ("entropy", "entropy"),
    ("in_history", "generated track heard before"),
    ("in_slate", "generated track already in the slate"),
    ("slate_artists", "distinct artists in the slate so far"),
    ("length", "length of the generated track, s"),
    ("log_count", "log10 train count of the generated track"),
    ("proxy_ratio", "proxy E[ratio] of the generated track"),
    ("gold_ratio", "gold E[ratio]"),
    ("gold_minus_proxy", "gold − proxy E[ratio]"),
    ("proxy_ratio_std", "proxy ensemble std"),
    ("gold_ratio_std", "gold ensemble std"),
]
STYLES = {"accept": "-", "expected": "--", "sampled": ":"}


def plot_steps(decoding: str) -> None:
    data = summary.filter(pl.col("decoding") == decoding)
    panels = [(col, title) for col, title in PANELS if col in data.columns]
    fig, axes = plt.subplots(3, 5, figsize=(26, 13))
    colours = {p: f"C{i}" for i, p in enumerate(data["policy"].unique(maintain_order=True))}
    for ax, (col, title) in zip(axes.ravel(), panels, strict=False):
        for (policy, mode), rows in data.group_by("policy", "mode", maintain_order=True):
            ax.plot(
                rows["step"],
                rows[col],
                STYLES[mode],
                marker="o",
                color=colours[policy],
                label=f"{policy} / {mode}",
            )
        ax.set(xlabel="step", title=title)
    for ax in axes.ravel()[len(panels) :]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(f"decoding: {decoding}")
    save_figure(fig, RESULTS / f"rollouts_{decoding}.png")
    plt.show()


for decoding in ("greedy", "sample"):
    plot_steps(decoding)